# Base 预训练数据清洗（6 步）
#
# ① AI 身份声明 — 模型名/自述/开发者声明 → 移除
# ② 代码/标记噪声 — LaTeX/Markdown表格/代码块 → 移除
# ③ 纯英文(无数学) — 英文文章/对话（不含数学内容）→ 移除
# ④ 非中英文语种 — 日/韩/俄/德/法等 → 移除
# ⑤ 中外文翻译 — 中↔英/日/韩等翻译指令 → 移除（文言文→现代汉语保留）
# ⑥ 文本规范化 — NBSP/多余空格/多余换行 → 清理后写入
#
# ★ 所有检测均用子串匹配，不用正则
# ★ 英文数学数据保留（含 equation/solve/formula 等数学关键词或运算符）
#
# 输出: pretrain_t2t_cleaned.jsonl / _removed.jsonl / _removed_stats.json

In [ ]:
import json
import os
import time
from array import array
from collections import defaultdict
from pathlib import Path

# 自动适配本地/DSW 路径
BASE = Path.cwd()
if not (BASE / "tokenizer_minimind_8k").exists() and (BASE / "shayler2.0").exists():
    BASE = BASE / "shayler2.0"
print(f"项目根目录: {BASE}")

# ============ CONFIG ============
INPUT_PATH = BASE / "data_stage1" / "pretrain_t2t.jsonl"           # 原始 Base 数据
CLEANED_PATH = BASE / "data_stage1" / "pretrain_t2t_cleaned.jsonl" # 清洗后
REMOVED_PATH = BASE / "data_stage1" / "pretrain_t2t_removed.jsonl" # 被移除数据
STATS_PATH = BASE / "data_stage1" / "pretrain_t2t_removed_stats.json"  # 统计

# 检查
print(f"输入: {INPUT_PATH} {'✅' if INPUT_PATH.exists() else '❌ 不存在'}")
if INPUT_PATH.exists():
    size_gb = INPUT_PATH.stat().st_size / 1e9
    print(f"      大小: {size_gb:.2f} GB")
print(f"输出: {CLEANED_PATH}")
print(f"移除: {REMOVED_PATH}")
print(f"统计: {STATS_PATH}")

In [ ]:
# ===== AI 身份关键词库 =====
IDENTITY_KEYWORDS = {
    "模型专名": [
        "通义千问", "Qwen", "文心一言", "ERNIE", "讯飞星火",
        "ChatGPT", "OpenAI", "GPT-4", "GPT-4o", "GPT-3.5", "GPT-3",
        "Claude", "Anthropic", "Kimi", "月之暗面",
        "DeepSeek", "深度求索", "Gemini",
        "百川智能", "Baichuan", "ChatGLM", "智谱清言", "MOSS",
        "SenseChat", "360智脑", "Llama", "Mistral", "Mixtral",
        "紫东太初", "书生浦语", "悟道", "盘古大模型",
        "MiniMax", "Yi-Large", "Yi-Chat",
        "腾讯混元", "腾讯元宝", "豆包", "天工AI",
    ],
    "AI身份声明": [
        "我是AI", "我是一个AI", "我是个AI", "我是人工智能", "我是一个人工智能",
        "我是语言模型", "我是大语言模型", "我是一个语言模型",
        "我是大模型", "我是一个大模型", "我是大型语言模型", "我是一个大型语言模型",
        "我是生成式AI", "我是生成式人工智能",
        "我是预训练语言模型", "我是基于Transformer的语言模型",
        "我是虚拟助手", "我是智能助手", "我是AI助手",
        "我是一个AI助手", "我是人工智能助手",
        "我是聊天机器人", "我是AI聊天机器人",
        "我是对话系统", "我是AI对话系统",
        "我是文本生成模型", "我是一个聊天程序", "我是AI程序", "我是一个AI程序",
        "我是你的AI", "我是你的智能", "我是你的AI助手", "我是你的智能助手",
        "作为AI", "作为一个AI", "作为一个人工智能", "作为人工智能",
        "身为AI", "身为一个人工智能", "身为人工智能",
        "因为我是AI", "由于我是AI", "因为我是人工智能",
        "因为我是语言模型", "由于我是语言模型",
        "作为语言模型", "作为一个语言模型", "身为语言模型",
        "作为大语言模型", "作为一个大语言模型",
        "我只是AI", "我只是一个AI", "我只是人工智能",
        "我只是个AI", "我只是个程序", "我只是一个程序", "我只是一个模型",
        "我仅仅是一个AI", "我仅仅是个AI", "我只是个语言模型", "我仅是一个语言模型",
        "我不是人类", "我不是真正的人", "我不是真实的人",
        "我并不是真正的人", "我并不是真实的人",
        "我不是真人", "我并不具有人类",
        "我不是一个真正的人", "我并非真实的人类",
        "我没有人类的", "我不具备人类",
        "我没有感情", "我没有情感", "我没有情绪",
        "我没有感受", "我没有意识", "我没有自我意识",
        "我没有实体", "我没有身体", "我没有物理形态",
        "我没有真实情感", "我没有人类情感",
        "我没有人类的感情", "我没有人的情感",
        "没有真实的感情", "没有真实的情感",
        "我不能像人类", "我无法像人类",
        "我不能体验", "我无法体验", "我无法感受",
        "我不能感受", "我不能理解情感",
        "我不能产生情感", "我无法产生情感",
        "我是一个被训练的", "我是一个被开发的",
        "我是被训练出来的", "我是被开发出来的",
        "我被设计用来帮助", "我被训练来帮助",
        "作为AI助手", "作为人工智能助手",
        "很高兴为你服务", "有什么我可以帮你的", "有什么我可以帮助你的",
    ],
    "开发者声明": [
        "我的创造者", "我的开发者", "我的设计者是",
        "由人工智能技术驱动",
        "我的知识截止于", "我的训练数据截止", "我的知识更新时间",
        "我是一个由", "由深度神经网络构成的",
    ],
}
ALL_KEYWORDS = [(kw, cat) for cat, kws in IDENTITY_KEYWORDS.items() for kw in kws]

QUICK_TRIGGERS = [
    "AI", "模型", "助手", "程序", "虚拟", "机器",
    "ChatGPT", "GPT", "Claude", "Qwen", "我是", "作为",
    "人工", "训练", "语言", "创造", "开发", "设计",
    "通义", "文心", "讯飞", "百川", "智谱", "DeepSeek",
    "Chat", "Kimi", "Gemini", "MOSS", "Llama", "豆包",
]

# ═══ 代码/标记/表格噪声 ═══
CODE_MARKERS = [
    "\\begin{", "\\end{", "\\frac", "\\sqrt", "\\sum", "\\int",
    "\\alpha", "\\beta", "\\gamma", "\\delta", "\\lambda",
    "\\mathbb", "\\mathcal", "\\mathbf", "\\text",
    "\\left", "\\right", "\\cdot", "\\times", "\\infty",
    "|---", "| ---", "|:---", "|---|", "| :---",
    "```", "<!--", "-->", "<?xml", "<!DOCTYPE", "<html", "</html",
    "&nbsp;", "&lt;", "&gt;", "&amp;", "&quot;", "&#",
    "import ", "def ", "class ", "function ",
    "const ", "let ", "var ", "print(", "return ",
    "===", "***", "----",
]
CODE_MARKER_THRESHOLD = 2
MATH_DOLLAR_THRESHOLD = 4
PIPE_THRESHOLD = 5
NEWLINE_RATIO_THRESHOLD = 0.08

def is_code_noise(text):
    if sum(1 for m in CODE_MARKERS if m in text) >= CODE_MARKER_THRESHOLD: return True
    if text.count("$") >= MATH_DOLLAR_THRESHOLD: return True
    if text.count("|") >= PIPE_THRESHOLD: return True
    if len(text) > 50 and text.count("\n") / len(text) > NEWLINE_RATIO_THRESHOLD: return True
    return False

# ═══ 英文检测（有数学保留，无数学移除）═══
CJK_MIN_RATIO = 0.02
ENGLISH_MIN_LEN = 100

def cjk_ratio(text):
    if not text: return 0.0
    cjk = sum(1 for c in text if (
        '一' <= c <= '鿿' or '㐀' <= c <= '䶿' or
        '豈' <= c <= '﫿' or '　' <= c <= '〿' or '＀' <= c <= '￯'))
    return cjk / len(text)

def has_math_content(text):
    tl = text.lower()
    if text.count('$') >= 2: return True
    B = chr(92)
    if any(c in text for c in [B+'frac',B+'sqrt',B+'sum',B+'int',B+'cdot',
        B+'times',B+'pi',B+'theta',B+'alpha',B+'beta',B+'gamma',B+'lambda',B+'infty',B+'partial']):
        return True
    math_kw = ['equation','solve','solving','formula','theorem','calculate','calculation',
        'derivative','integral','polynomial','matrix','vector','algebra','geometry',
        'trigonometry','calculus','fraction','decimal','exponent','logarithm','quadratic',
        'linear equation','probability','statistics','graph of','slope of','perpendicular',
        'parallel lines','right triangle','pythagorean','circumference','diameter','radius',
        'arithmetic','find the value','what is the sum','evaluate the','simplify the',
        'compute the','solve for x','solve for y','find x','find y']
    if sum(1 for kw in math_kw if kw in tl) >= 1: return True
    import re
    if len(re.findall(r'\d[\d\s.xXyYzZ]*[+\-*/×÷][\d\s.xXyYzZ]*=', text)) >= 2: return True
    if len(text) > 0 and sum(1 for c in text if c in '=+-*/^') / len(text) > 0.03: return True
    return False

def is_non_math_english(text):
    if len(text) < ENGLISH_MIN_LEN: return False
    if cjk_ratio(text) >= CJK_MIN_RATIO: return False
    if has_math_content(text): return False
    return True

# ═══ 非中英文语种检测（日/韩/俄/德/法…）═══
def has_foreign_script(text):
    for c in text:
        cp = ord(c)
        if 0x3040 <= cp <= 0x309F: return True  # 平假名
        if 0x30A0 <= cp <= 0x30FF: return True  # 片假名
        if 0xAC00 <= cp <= 0xD7AF: return True  # 韩文
        if 0x1100 <= cp <= 0x11FF: return True  # 韩文字母
        if 0x0400 <= cp <= 0x04FF: return True  # 西里尔
        if 0x0600 <= cp <= 0x06FF: return True  # 阿拉伯
        if 0x0E00 <= cp <= 0x0E7F: return True  # 泰文
        if 0x0900 <= cp <= 0x097F: return True  # 天城文
        if 0x0590 <= cp <= 0x05FF: return True  # 希伯来
    return False

def foreign_char_ratio(text):
    if not text: return 0.0
    foreign = 0
    for c in text:
        cp = ord(c)
        if ('一' <= c <= '鿿' or '㐀' <= c <= '䶿' or
            '豈' <= c <= '﫿' or '　' <= c <= '〿' or '＀' <= c <= '￯'):
            continue
        if cp < 128: continue
        if 0x00C0 <= cp <= 0x00FF: foreign += 1; continue
        if cp >= 128: foreign += 1
    return foreign / len(text)

FOREIGN_CHAR_THRESHOLD = 0.05

def is_foreign_language(text):
    if has_foreign_script(text): return True
    if foreign_char_ratio(text) > FOREIGN_CHAR_THRESHOLD: return True
    return False

# ═══ 翻译数据检测（仅移除中外文翻译，文言文保留）═══
TRANSLATION_TRIGGERS = [
    "翻译成", "翻译为", "翻译下列", "翻译以下",
    "请翻译", "把下列", "把以下", "将下列", "将以下",
    "译成", "译为", "请将下列", "请将以下", "请把下列", "请把以下",
    "translate the following", "translate this",
    "translate into", "translation:",
    "原文：", "译文：", "原文:", "译文:",
    "英文原文", "中文译文", "参考译文",
]
FOREIGN_LANG_NAMES = [
    "英文", "英语", "英文版", "英译", "日文", "日语", "日译",
    "韩文", "韩语", "韩译", "法文", "法语", "法译",
    "德文", "德语", "德译", "俄文", "俄语", "俄译",
    "西班牙语", "西班牙文", "阿拉伯语", "阿拉伯文", "葡萄牙语", "意大利语",
    "English", "Japanese", "Korean", "French", "German",
    "Russian", "Spanish", "Arabic", "Portuguese",
    "in English", "in Japanese", "in French", "in German",
]
CLASSICAL_CN_MARKERS = [
    "文言文", "古文", "古诗", "古诗词", "古汉语", "文言",
    "唐诗", "宋词", "元曲", "诗经", "论语", "史记",
    "世说新语", "古文观止", "三字经", "千字文",
    "现代汉语", "现代文", "白话文", "白话", "用现代", "译为现代",
]

def is_translation_data(text):
    tl = text.lower()
    if any(m in text for m in CLASSICAL_CN_MARKERS): return False
    if not any(kw in tl for kw in TRANSLATION_TRIGGERS): return False
    if any(kw in tl for kw in FOREIGN_LANG_NAMES): return True
    lang_tags = ["英文：","中文：","英文:","中文:","English：","Chinese：","英语：","汉语："]
    if sum(1 for t in lang_tags if t in text) >= 2: return True
    return False

# ═══ 文本规范化 ═══
def normalize_text(text):
    B = chr(92)  # 反斜杠，避免 JSON 转义歧义
    text = text.replace(" ", " ")        # Unicode NBSP → 空格
    text = text.replace(B + B, B)             # ★ 双反斜杠 → 单反斜杠（必须在最前）
    text = text.replace(B + 'n', '\n')        # ★ 字面 \n → 真正换行
    text = text.replace(B + 't', '\t')        # ★ 字面 \t → 真正制表符
    text = text.replace(B + 'r', '\r')        # ★ 字面 \r → 真正回车
    text = text.replace(B + '"', '"')         # ★ 转义引号 → 真正引号
    text = text.replace("&nbsp;", " ")        # HTML 实体
    text = text.replace("&amp;", "&")
    text = text.replace("&lt;", "<")
    text = text.replace("&gt;", ">")
    text = text.replace("&quot;", '"')
    while "\n\n\n" in text: text = text.replace("\n\n\n", "\n\n")
    lines = []
    for line in text.split("\n"):
        while "  " in line: line = line.replace("  ", " ")
        lines.append(line.strip())
    return "\n".join(lines).strip()

print(f"AI 身份关键词: {len(ALL_KEYWORDS)} 个")
for cat, kws in IDENTITY_KEYWORDS.items():
    print(f"  {cat}: {len(kws)} 个")
print(f"代码标记: {len(CODE_MARKERS)} 个")
print(f"外语检测: 非中英文字符 > {FOREIGN_CHAR_THRESHOLD:.0%} → 移除")
print(f"翻译检测: 中外文翻译 → 移除 | 文言文翻译 → 保留")

In [ ]:
# ===== 执行清洗（6 步）=====
# ① AI 身份  ② 代码噪声  ③ 纯英文(无数学)
# ④ 非中英文语种  ⑤ 中外文翻译  ⑥ 文本规范化 → 写入

def quick_scan(text):
    tl = text.lower()
    for t in QUICK_TRIGGERS:
        if t.lower() in tl: return True
    return False

def match_identity(text):
    tl = text.lower()
    for kw, cat in ALL_KEYWORDS:
        if kw.lower() in tl: return True, cat, kw
    return False, None, None

if CLEANED_PATH.exists():
    print(f"✅ 清洗后文件已存在: {CLEANED_PATH.name}")
    print(f"   如需重新清洗请先删除此文件")
else:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"❌ 原始数据不存在: {INPUT_PATH}")

    print(f"开始清洗: {INPUT_PATH.name}")
    print(f"大小: {INPUT_PATH.stat().st_size / 1e9:.2f} GB\n")

    total = 0
    r_identity = r_code = r_english = r_foreign = r_trans = r_other = 0
    normalized = 0
    removed_lines, cat_counter, kw_counter, cat_samples = [], defaultdict(int), defaultdict(int), defaultdict(list)
    t0 = time.time()

    with open(INPUT_PATH, "r", encoding="utf-8") as fin, \
         open(CLEANED_PATH, "w", encoding="utf-8") as fout:
        for line in fin:
            ls = line.strip()
            if not ls: continue
            total += 1
            try:
                obj = json.loads(ls)
                text = obj.get("text", "")
            except json.JSONDecodeError:
                r_other += 1; continue
            if not text or len(text) < 10:
                r_other += 1; continue

            # ── ① AI 身份 ──
            if quick_scan(text):
                matched, cat, kw = match_identity(text)
                if matched:
                    r_identity += 1
                    cat_counter[cat] += 1; kw_counter[kw] += 1
                    removed_lines.append({"line_no": total, "category": cat, "keyword": kw, "reason": "AI身份", "text": text[:300]})
                    if len(cat_samples[cat]) < 3: cat_samples[cat].append((kw, text[:200]))
                    continue

            # ── ② 代码/标记/表格 ──
            if is_code_noise(text):
                r_code += 1; cat_counter["代码噪声"] += 1
                removed_lines.append({"line_no": total, "category": "代码噪声", "keyword": "", "reason": "代码/标记/表格", "text": text[:300]})
                if len(cat_samples["代码噪声"]) < 3: cat_samples["代码噪声"].append(("", text[:200]))
                continue

            # ── ③ 纯英文（无数学）──
            if is_non_math_english(text):
                r_english += 1; cat_counter["纯英文"] += 1
                removed_lines.append({"line_no": total, "category": "纯英文", "keyword": "", "reason": "纯英文(无数学)", "text": text[:300]})
                if len(cat_samples["纯英文"]) < 3: cat_samples["纯英文"].append(("", text[:200]))
                continue

            # ── ④ 非中英文语种 ──
            if is_foreign_language(text):
                r_foreign += 1; cat_counter["非中英文语种"] += 1
                removed_lines.append({"line_no": total, "category": "非中英文语种", "keyword": "", "reason": "日/韩/俄/德/法等", "text": text[:300]})
                if len(cat_samples["非中英文语种"]) < 3: cat_samples["非中英文语种"].append(("", text[:200]))
                continue

            # ── ⑤ 中外文翻译 ──
            if is_translation_data(text):
                r_trans += 1; cat_counter["中外文翻译"] += 1
                removed_lines.append({"line_no": total, "category": "中外文翻译", "keyword": "", "reason": "翻译数据", "text": text[:300]})
                if len(cat_samples["中外文翻译"]) < 3: cat_samples["中外文翻译"].append(("", text[:200]))
                continue

            # ── ⑥ 规范化 + 写入 ──
            ct = normalize_text(text)
            if ct != text: normalized += 1
            if len(ct) >= 10:
                obj["text"] = ct
                fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            else:
                r_other += 1

            if total % 500000 == 0:
                e = time.time() - t0
                print(f"  {total/1e6:.1f}M | AI身份 {r_identity:,} | 代码 {r_code:,} | "
                      f"英文 {r_english:,} | 外语 {r_foreign:,} | 翻译 {r_trans:,} | {e:.0f}s")

    elapsed = time.time() - t0
    removed_total = r_identity + r_code + r_english + r_foreign + r_trans + r_other
    kept = total - removed_total

    # ── 写出移除数据 ──
    with open(REMOVED_PATH, "w", encoding="utf-8") as f:
        for item in removed_lines: f.write(json.dumps(item, ensure_ascii=False) + "\n")

    # ── 统计 ──
    stats = {"input_file": str(INPUT_PATH), "total_lines": total,
        "removed_total": removed_total, "removed_pct": round(removed_total/total*100,2) if total else 0,
        "r_identity": r_identity, "r_code": r_code, "r_english": r_english,
        "r_foreign": r_foreign, "r_trans": r_trans, "r_other": r_other,
        "normalized": normalized, "kept": kept,
        "categories": dict(cat_counter),
        "top_keywords": sorted(kw_counter.items(), key=lambda x: x[1], reverse=True)[:30],
        "elapsed_seconds": round(elapsed, 1)}
    with open(STATS_PATH, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    # ── 报告 ──
    print(f"\n{'='*60}")
    print(f"清洗完成 — {elapsed:.0f}s")
    print(f"{'='*60}")
    print(f"  总行数:         {total:,}")
    print(f"  ① AI 身份:     {r_identity:>8,} ({r_identity/total*100:.1f}%)")
    print(f"  ② 代码噪声:    {r_code:>8,} ({r_code/total*100:.1f}%)")
    print(f"  ③ 纯英文:      {r_english:>8,} ({r_english/total*100:.1f}%)")
    print(f"  ④ 非中英文语种:{r_foreign:>8,} ({r_foreign/total*100:.1f}%)")
    print(f"  ⑤ 中外文翻译:  {r_trans:>8,} ({r_trans/total*100:.1f}%)")
    print(f"  ⑥ 其他:        {r_other:>8,}")
    print(f"  文本规范化:    {normalized:>8,}")
    print(f"  ─────────────────────")
    print(f"  保留:          {kept:>8,} ({kept/total*100:.1f}%)")

    print(f"\n按分类统计:")
    for cat in sorted(cat_counter.keys(), key=lambda c: cat_counter[c], reverse=True):
        cnt = cat_counter[cat]
        print(f"  {cat:<16} {cnt:>8,} {cnt/removed_total*100:>7.1f}%" if removed_total else f"  {cat:<16} {cnt:>8,}")

    if kw_counter:
        print(f"\nAI 身份关键词 Top 20:")
        for kw, cnt in sorted(kw_counter.items(), key=lambda x: x[1], reverse=True)[:20]:
            print(f"  {kw:<30} {cnt:>8,}")

    for cat in sorted(cat_counter.keys(), key=lambda c: cat_counter[c], reverse=True):
        print(f"\n[{cat}] 样本:")
        for kw, s in cat_samples[cat][:2]:
            print(f"  {s[:150]}...")

    in_gb = INPUT_PATH.stat().st_size / 1e9
    out_gb = CLEANED_PATH.stat().st_size / 1e9
    print(f"\n文件: {in_gb:.2f} GB → {out_gb:.2f} GB")
    print(f"✅ {CLEANED_PATH.name}")
    print(f"✅ {REMOVED_PATH.name} ({len(removed_lines):,} 条)")
    print(f"✅ {STATS_PATH.name}")

In [ ]:
# ===== Cell 3.5：逐行原始文本匹配（不做 JSON 解析，命中即删）=====
# 解决 Cell 3 的 JSON 解析 + 字段提取导致的遗漏问题
# 直接对整行原始文本做子串匹配（kw.lower() in line.lower()），彻底清除身份残留

SRC_V2 = BASE / "data_stage1" / "pretrain_t2t_cleaned_two.jsonl"  # Cell 3 输出
DST_V2 = BASE / "data_stage1" / "pretrain_t2t_cleaned_v2.jsonl"   # 最终清洗

KW_V2 = [
    # 模型名
    '通义千问','Qwen','qwen','文心一言','ERNIE','讯飞星火',
    'ChatGPT','chatgpt','OpenAI','GPT-4','GPT-4o','GPT-3','GPT-3.5',
    'Claude','claude','Anthropic','Kimi','kimi','月之暗面',
    'DeepSeek','deepseek','深度求索','Gemini','gemini',
    '百川智能','Baichuan','ChatGLM','智谱清言','智谱AI','MOSS','moss',
    'SenseChat','360智脑','Llama','llama','Mistral','Mixtral',
    '紫东太初','书生浦语','悟道','盘古大模型','MiniMax','Yi-Large','Yi-Chat',
    '腾讯混元','腾讯元宝','豆包','天工AI',
    # AI 身份声明
    '我是AI','我是一个AI','我是个AI','我是人工智能','我是一个人工智能',
    '我是语言模型','我是大语言模型','我是一个语言模型',
    '我是大模型','我是一个大模型','我是大型语言模型','我是一个大型语言模型',
    '我是生成式AI','我是生成式人工智能',
    '我是虚拟助手','我是智能助手','我是AI助手',
    '我是一个AI助手','我是人工智能助手',
    '我是聊天机器人','我是AI聊天机器人',
    '我是对话系统','我是AI对话系统',
    '我是文本生成模型','我是一个聊天程序','我是AI程序','我是一个AI程序',
    '我是你的AI','我是你的智能','我是你的AI助手','我是你的智能助手',
    '作为AI','作为一个AI','作为一个人工智能','作为人工智能',
    '身为AI','身为一个人工智能','身为人工智能',
    '因为我是AI','由于我是AI','因为我是人工智能',
    '因为我是语言模型','由于我是语言模型',
    '作为语言模型','作为一个语言模型','身为语言模型',
    '作为大语言模型','作为一个大语言模型',
    '我只是AI','我只是一个AI','我只是人工智能',
    '我只是个AI','我只是个程序','我只是一个程序','我只是一个模型',
    '我仅仅是一个AI','我仅仅是个AI','我只是个语言模型',
    '我不是人类','我不是真正的人','我不是真实的人',
    '我并不是真正的人','我并不具有人类',
    '我没有人类的','我不具备人类',
    '我没有感情','我没有情感','我没有情绪',
    '我没有感受','我没有意识','我没有自我意识',
    '我没有实体','我没有身体','我没有物理形态',
    '我没有真实情感','我没有人类情感',
    '我没有人类的感情','我没有人的情感',
    '没有真实的感情','没有真实的情感',
    '我不能像人类','我无法像人类',
    '我不能体验','我无法体验','我无法感受',
    '我不能感受','我不能理解情感','我不能产生情感',
    '我是一个被训练的','我是一个被开发的',
    '我是被训练出来的','我是被开发出来的',
    '我被设计用来帮助','我被训练来帮助',
    '作为AI助手','作为人工智能助手',
    '我的创造者','我的开发者','我的设计者是',
    '由人工智能技术驱动',
    '我的知识截止于','我的训练数据截止','我的知识更新时间',
    '我是一个由','由深度神经网络构成的',
    '很高兴为你服务','有什么我可以帮你的',
    'jingyaogong','minimind',
]

print(f"Cell 3.5：逐行原始文本匹配（不做 JSON 解析）")
print(f"输入: {SRC_V2.name} {'✅' if SRC_V2.exists() else '❌'}")
print(f"输出: {DST_V2.name}")
print(f"关键词: {len(KW_V2)} 个")

total = 0
removed = 0
kc = defaultdict(int)

with open(SRC_V2, 'r', encoding='utf-8') as fin, open(DST_V2, 'w', encoding='utf-8') as fout:
    for line in fin:
        if not line.strip():
            continue
        total += 1
        ll = line.lower()
        hit = None
        for kw in KW_V2:
            if kw.lower() in ll:
                hit = kw
                break
        if hit:
            removed += 1
            kc[hit] += 1
        else:
            fout.write(line)

        if total % 500000 == 0:
            print(f"  {total/1e6:.1f}M | 删除 {removed:,} ({removed/total*100:.2f}%)")

print(f"\n总 {total:,} | 删除 {removed:,} ({removed/total*100:.2f}%) | 保留 {total-removed:,}")

if kc:
    print(f"\n命中关键词 Top 20:")
    for kw, cnt in sorted(kc.items(), key=lambda x: x[1], reverse=True)[:20]:
        print(f"  {kw}: {cnt}")
else:
    print("✅ 无任何 AI 身份关键词命中")

print(f"\n✅ Cell 3.5 完成: {DST_V2.name}")

In [ ]:
# ===== Cell 3.6：转义残留修复（独立运行，不依赖 Cell 2/3/3.5）=====
# 修复 JSON 文本中字面 \n \t \r \" \\ → 对应真实字符
# 可直接重跑本 Cell，无需重跑全流程
# ★ 使用 chr(92) 避免反斜杠转义问题

SRC_ESC = BASE / "data_stage1" / "pretrain_t2t_cleaned_v2.jsonl"    # Cell 3.5 输出
DST_ESC = BASE / "data_stage1" / "pretrain_t2t_cleaned_v2_fixed.jsonl"

def fix_escapes(text):
    """转义残留修复，顺序敏感。使用 chr(92)=反斜杠 避免转义歧义"""
    B = chr(92)  # 反斜杠 \
    text = text.replace(B + B, B)        # ① \\ → \（必须在最前）
    text = text.replace(B + 'n', '\n')   # ② \n → 真正换行
    text = text.replace(B + 't', '\t')   # ③ \t → 真正制表符
    text = text.replace(B + 'r', '\r')   # ④ \r → 真正回车
    text = text.replace(B + '"', '"')    # ⑤ \" → 真正引号
    return text

print(f"Cell 3.6：转义残留修复（独立运行）")
print(f"输入: {SRC_ESC.name} {'✅' if SRC_ESC.exists() else '❌'}")
print(f"输出: {DST_ESC.name}")

total = 0
fixed = 0

with open(SRC_ESC, 'r', encoding='utf-8') as fin, open(DST_ESC, 'w', encoding='utf-8') as fout:
    for line in fin:
        if not line.strip():
            continue
        total += 1
        try:
            obj = json.loads(line)
            text = obj.get('text', '')
        except json.JSONDecodeError:
            continue

        fixed_text = fix_escapes(text)
        if fixed_text != text:
            fixed += 1
        obj['text'] = fixed_text
        fout.write(json.dumps(obj, ensure_ascii=False) + '\n')

        if total % 500000 == 0:
            print(f"  {total/1e6:.1f}M | 修复 {fixed:,} ({fixed/total*100:.2f}%)")

print(f"\n总 {total:,} | 修复 {fixed:,} ({fixed/total*100:.2f}%) | 未改 {total-fixed:,}")
print(f"✅ Cell 3.6 完成: {DST_ESC.name}")

In [ ]:
# ===== 快速抽查：随机预览清洗后数据 =====
import random

n_sample = 10
print(f"从清洗后文件随机抽取 {n_sample} 条预览:\n")

with open(CLEANED_PATH, "r", encoding="utf-8") as f:
    all_clean = f.readlines()

if len(all_clean) > n_sample:
    samples = random.sample(all_clean, n_sample)
else:
    samples = all_clean

for i, line in enumerate(samples):
    try:
        obj = json.loads(line.strip())
        text = obj.get("text", "")
        print(f"[{i}] {text[:150]}{'...' if len(text) > 150 else ''}")
    except Exception:
        pass

# 验证没有身份泄露（用几个最不可能误伤的关键词做快速扫描）
print(f"\n身份泄露复查（全量 {len(all_clean):,} 条）...")
leak_checks = [
    ("通义千问", "通义千问"),
    ("ChatGPT", "ChatGPT"),
    ("我是AI", "我是AI"),
    ("作为人工智能", "作为人工智能"),
    ("DeepSeek", "DeepSeek"),
    ("MOSS", "MOSS"),
]
found_leaks = defaultdict(int)
for line in all_clean:
    try:
        text = json.loads(line.strip()).get("text", "")
        for kw, label in leak_checks:
            if kw in text:
                found_leaks[label] += 1
    except Exception:
        pass

if found_leaks:
    print("⚠️ 仍存在以下身份关键词:")
    for label, cnt in sorted(found_leaks.items(), key=lambda x: x[1], reverse=True):
        print(f"  {label}: {cnt} 条")
else:
    print("✅ 未发现已知身份关键词")

In [ ]:
# ===== 复查 ①：已知关键词重扫清洗后数据 =====
# 用完全相同的关键词扫描清洗后文件，验证 0 命中
print("=" * 50)
print("复查 ①：已知关键词重扫")
print("=" * 50)

if not CLEANED_PATH.exists():
    print("❌ 清洗后文件不存在，请先运行 Cell 3")
else:
    clean_hits = 0
    clean_hit_samples = []
    clean_total = 0

    with open(CLEANED_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                text = json.loads(line).get("text", "")
            except json.JSONDecodeError:
                continue
            clean_total += 1

            matched, cat, kw = match_identity(text)
            if matched:
                clean_hits += 1
                if len(clean_hit_samples) < 10:
                    clean_hit_samples.append(f"  [{cat}] 命中: {kw} → {text[:150]}...")

    print(f"扫描: {clean_total:,} 条")
    if clean_hits == 0:
        print(f"✅ 已知关键词命中: 0 条——清洗无遗漏")
    else:
        print(f"❌ 命中: {clean_hits} 条（清洗遗漏！）")
        for s in clean_hit_samples:
            print(s)

In [ ]:
# ===== 复查 ②：可疑遗漏扫描 =====
# 从清洗后数据中找"通过预筛但未命中关键词"的条目
# 这些可能是关键词库没覆盖到的身份表达变体
print("\n" + "=" * 50)
print("复查 ②：可疑遗漏扫描")
print("=" * 50)

if not CLEANED_PATH.exists():
    print("❌ 清洗后文件不存在")
else:
    # 第一人称身份相关关键词（比 QUICK_TRIGGERS 更窄，定位疑似身份文本）
    SUSPECT_KW = [
        "我是", "我叫", "我的名字", "我是一个",
        "AI", "人工智能", "语言模型", "大模型",
        "助手", "机器人", "程序", "自动回复",
        "虚拟", "生成", "训练", "开发",
    ]
    SUSPECT_LIMIT = 1000

    suspect_lines = []
    scan_total = 0

    with open(CLEANED_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            scan_total += 1

            try:
                text = json.loads(line).get("text", "")
            except json.JSONDecodeError:
                continue

            # 包含 ≥2 个可疑词，且未被已知关键词命中
            kw_hits = sum(1 for kw in SUSPECT_KW if kw in text)
            if kw_hits >= 2 and not match_identity(text)[0]:
                suspect_lines.append({
                    "line_no": scan_total,
                    "kw_count": kw_hits,
                    "text": text[:250]
                })
                if len(suspect_lines) >= SUSPECT_LIMIT:
                    break

    print(f"扫描: {scan_total:,} 条（截止到第 {SUSPECT_LIMIT} 条疑似）")
    print(f"疑似未命中: {len(suspect_lines)} 条（含 ≥2 个可疑词但关键词未命中）")

    if suspect_lines:
        n_show = min(30, len(suspect_lines))
        print(f"\n随机抽查 {n_show} 条疑似样本（人工判断是否需要补充关键词）:\n")
        for s in random.sample(suspect_lines, n_show):
            print(f"  [{s['line_no']}] (命中 {s['kw_count']} 个可疑词)")
            print(f"    {s['text'][:180]}...")
            print()

        SUSPECT_PATH = BASE / "data_stage1" / "pretrain_t2t_suspect.jsonl"
        with open(SUSPECT_PATH, "w", encoding="utf-8") as f:
            for item in suspect_lines:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        print(f"✅ 全部 {len(suspect_lines)} 条疑似样本已保存: {SUSPECT_PATH.name}")
    else:
        print("✅ 未发现疑似遗漏")

In [ ]:
# ===== 复查 ③：反向分析——从被移除数据找模式缺口 =====
# 分析被移除数据的高频模式，反推是否有变体未覆盖
import re as re_mod

print("\n" + "=" * 50)
print("复查 ③：反向分析")
print("=" * 50)

if not REMOVED_PATH.exists():
    print("❌ 被移除数据文件不存在，请先运行 Cell 3")
else:
    removed_texts = []
    removed_kws = []
    with open(REMOVED_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line.strip())
                removed_texts.append(item.get("text", ""))
                removed_kws.append(item.get("keyword", ""))
            except json.JSONDecodeError:
                pass

    print(f"分析 {len(removed_texts):,} 条被移除文本...")

    # ── 被移除数据中的高频 2-4 字短语 ──
    STOP = {"一个", "一种", "可以", "没有", "不是", "这个", "那个", "什么",
            "我们", "他们", "自己", "因为", "所以", "但是", "如果", "虽然",
            "而且", "或者", "已经", "还是", "只是", "就是", "不过", "吗",
            "呢", "吧", "啊", "的", "了", "在", "是", "有", "和", "就",
            "都", "也", "要", "会", "能", "对", "等", "这", "那", "之"}

    phrase_counter = defaultdict(int)
    for text in removed_texts:
        chars = list(text)
        for n in [2, 3, 4]:
            for i in range(len(chars) - n + 1):
                phrase = "".join(chars[i:i+n])
                if phrase in STOP:
                    continue
                if re_mod.match(r'^[\s，。！？、；：""''（）　]+$', phrase):
                    continue
                phrase_counter[phrase] += 1

    AI_KW_SET = {"AI", "人工", "模型", "助手", "程序", "机器", "虚拟", "训练",
                 "语言", "智能", "对话", "聊天", "算法", "生成", "文本", "学习"}

    ai_phrases = [(p, c) for p, c in phrase_counter.items()
                  if any(kw in p for kw in AI_KW_SET) and c >= 10]
    ai_phrases.sort(key=lambda x: x[1], reverse=True)

    print(f"\n高频 AI 相关短语 (Top 30):")
    print(f"  {'短语':<20} {'频次':>6}")
    print(f"  {'-'*26}")
    for phrase, cnt in ai_phrases[:30]:
        print(f"  {phrase:<20} {cnt:>6}")

    # ── 被移除文本中"我..."开头片段 ──
    i_sentences = []
    for text in removed_texts:
        for m in re_mod.finditer(r'我[^，。！？；\n]{2,50}', text):
            i_sentences.append(m.group())
            if len(i_sentences) >= 500:
                break
        if len(i_sentences) >= 500:
            break

    print(f"\n被移除文本中'我...'开头片段 (随机 20 条):")
    for s in random.sample(i_sentences, min(20, len(i_sentences))):
        print(f"  • {s}")

    # ── 漏网关键词检查 ──
    TEST_KW = ["人工智能", "AI助手", "语言模型", "虚拟助手", "对话系统",
               "大模型", "LLM", "生成式", "深度学习", "自然语言",
               "MOSS", "通义", "文心", "ChatGPT", "Claude", "豆包",
               "自动回复", "自动生成", "算法生成", "大语言模型"]

    clean_sample = 0
    kw_in_clean = defaultdict(int)
    with open(CLEANED_PATH, "r", encoding="utf-8") as f:
        for line in f:
            clean_sample += 1
            if clean_sample >= 2_000_000:
                break
            try:
                text = json.loads(line.strip()).get("text", "")
                for kw in TEST_KW:
                    if kw in text:
                        kw_in_clean[kw] += 1
            except json.JSONDecodeError:
                pass

    print(f"\n潜在遗漏关键词检查（removed 中高频 vs clean 前 200 万行残留）:")
    print(f"  {'关键词':<14} {'removed中':>8} {'clean中':>8} {'状态':>6}")
    print(f"  {'-'*40}")
    for kw in TEST_KW:
        in_removed = sum(1 for t in removed_texts if kw in t)
        in_clean = kw_in_clean.get(kw, 0)
        if in_removed > 0 and in_clean > 10:
            status = "⚠️ 残留"
        elif in_removed > 0:
            status = "✅"
        else:
            status = "—"
        print(f"  {kw:<14} {in_removed:>8,} {in_clean:>8,} {status:>6}")

    if any(kw_in_clean.get(kw, 0) > 10 for kw in TEST_KW):
        print(f"\n⚠️ 上述标记 '⚠️ 残留' 的关键词在清洗后数据中仍有较多出现，建议补充到关键词库")
    else:
        print(f"\n✅ 所有检测关键词在清洗后数据中残留量 ≤ 10（可忽略）")

In [ ]:
# ===== 分词器加载 + 配置 =====
# 使用 Cell 3.6 转义修复后数据

import torch
from tokenizers import Tokenizer

TOKENIZER_PATH = BASE / "tokenizer_minimind_8k" / "tokenizer.json"
INPUT_FILE = BASE / "data_stage1" / "pretrain_t2t_cleaned_v2_fixed.jsonl"  # ★ Cell 3.6 输出
OUTPUT_DIR = BASE / "base_data" / "pretrain_pt_cleaned_v2"            # ★ v5.1 训练用
BLOCK_SIZE = 1024
SAMPLES_PER_FILE = 20000    # 每个 .pt 文件多少块

# BOS/EOS token IDs（MiniMind 8K 分词器）
BOS = 1   # <s>
EOS = 2   # </s>

print(f"分词器: {TOKENIZER_PATH} {'✅' if TOKENIZER_PATH.exists() else '❌ 不存在'}")
print(f"输入:   {INPUT_FILE} {'✅' if INPUT_FILE.exists() else '❌ 不存在'}")
if INPUT_FILE.exists():
    print(f"        大小: {INPUT_FILE.stat().st_size / 1e9:.2f} GB")
print(f"输出:   {OUTPUT_DIR}")

# 加载分词器
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
print(f"\n词表大小: {tokenizer.get_vocab_size()}")
print(f"BOS token: {BOS} (<s>)")
print(f"EOS token: {EOS} (</s>)")

# 验证中文 + BOS/EOS 编码
test_texts = [
    "你好，今天天气怎么样？",
    "人工智能是计算机科学的一个分支。",
    "柳小乐是一个温暖友好的AI伙伴。",
]
print(f"\n编码验证:")
for text in test_texts:
    ids = tokenizer.encode(text).ids
    has_unk = 0 in ids
    status = "❌ 含<unk>" if has_unk else "✅"
    print(f"  {status} '{text[:40]}...' → {len(ids)} tokens")

bos_ids = tokenizer.encode("<s>").ids
eos_ids = tokenizer.encode("</s>").ids
print(f"\n<s> 编码: {bos_ids}  (期望 [1])")
print(f"</s> 编码: {eos_ids}  (期望 [2])")
print(f"BOS 验证: {bos_ids == [BOS]}, EOS 验证: {eos_ids == [EOS]}")

In [ ]:
# ===== 分词 + 拼接 + 切块 + 存盘 =====
# 流程：逐行读取 → BOS + tokens + EOS → array("i") 拼接 → 切块 → 分片 .pt
# 无 loss mask、无角色标签——Base 预训练所有 token 平等参与
# 参考：preprocess_pretrain_data.ipynb Cell 6

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_tokens = array("i")  # 4 字节/int，比 Python list 省 ~7x 内存
total_lines = 0
skipped = 0

t0 = time.time()
print("🔪 分词中...\n")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
            text = obj.get("text", "")
        except json.JSONDecodeError:
            skipped += 1
            continue

        if not text or len(text) < 10:
            skipped += 1
            continue

        # BOS + tokens + EOS
        ids = tokenizer.encode(text).ids
        all_tokens.append(BOS)
        all_tokens.extend(ids)
        all_tokens.append(EOS)

        total_lines += 1
        if total_lines % 200000 == 0:
            elapsed = time.time() - t0
            print(f"  {total_lines:,} 行 | {len(all_tokens)/1e9:.2f}B tokens | {elapsed:.0f}s")

elapsed = time.time() - t0
print(f"  ✅ 分词完成: {total_lines:,} 行 | {len(all_tokens)/1e9:.2f}B tokens | {elapsed:.0f}s")
if skipped:
    print(f"  跳过空/短文本/解析错误: {skipped:,}")

# ===== 切块 =====
total_blocks = len(all_tokens) // BLOCK_SIZE
dropped = len(all_tokens) - total_blocks * BLOCK_SIZE
print(f"\n📦 切块: {total_blocks:,} 块 × {BLOCK_SIZE} tokens")
if len(all_tokens) > 0:
    print(f"   丢弃尾部: {dropped} tokens ({dropped/len(all_tokens)*100:.2f}%)")

# ===== 分片保存 =====
num_files = (total_blocks + SAMPLES_PER_FILE - 1) // SAMPLES_PER_FILE
print(f"\n💾 保存 {num_files} 个 .pt 文件...\n")

for i in range(num_files):
    start = i * SAMPLES_PER_FILE
    end = min(start + SAMPLES_PER_FILE, total_blocks)
    chunk_tokens = all_tokens[start * BLOCK_SIZE : end * BLOCK_SIZE]

    # 转 2D tensor: [N, block_size]
    chunk = torch.tensor(list(chunk_tokens), dtype=torch.int32).view(-1, BLOCK_SIZE)

    filepath = OUTPUT_DIR / f"pretrain_{i:04d}.pt"
    torch.save(chunk, filepath)

    size_mb = filepath.stat().st_size / 1024 / 1024
    print(f"  [{i+1:03d}/{num_files}] {filepath.name}  → {chunk.shape[0]:,} 块, {size_mb:.0f} MB")

elapsed_total = time.time() - t0
print(f"\n✅ 完成: {total_lines:,} 行 → {total_blocks:,} 块 → {num_files} 个 .pt 文件 | {elapsed_total:.0f}s")
print(f"输出: {OUTPUT_DIR}/")

In [ ]:
# ===== 最终统计 + 样本验证 =====
# 参考：preprocess_pretrain_data.ipynb Cell 7

pt_files = sorted(OUTPUT_DIR.glob("pretrain_*.pt"))
total_size = sum(f.stat().st_size for f in pt_files)

# 统计总块数
total_samples = 0
for fp in pt_files:
    data = torch.load(fp, map_location="cpu")
    total_samples += data.shape[0]
del data

print(f"{'='*50}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"总块数:   {total_samples:,}")
print(f"文件数:   {len(pt_files)}")
print(f"总大小:   {total_size/1024**3:.2f} GB")
print(f"输入行数: {total_lines:,}")
print(f"每行平均块数: {total_samples/total_lines:.2f}")

# 解码第一条验证
sample = torch.load(pt_files[0], map_location="cpu")
print(f"\n示例: {pt_files[0].name}")
print(f"  shape: {list(sample.shape)}, dtype={sample.dtype}")

t = sample[0].tolist()
# 解码（跳过 padding 0）
decoded = tokenizer.decode([x for x in t if x > 0])
print(f"  块大小: {BLOCK_SIZE}")
print(f"  解码前200字符: {decoded[:200]}...")

# 身份泄露检查
identity_kw = ["我是AI", "我是MOSS", "作为AI", "人工智能助手", "我没有感情"]
found_kw = [kw for kw in identity_kw if kw in decoded]
if found_kw:
    print(f"\n⚠️ 身份泄露: 发现关键词 {found_kw}")
else:
    print(f"\n✅ 身份泄露检查通过（首块样本）")

# 解码最后一块
t_last = sample[-1].tolist()
decoded_last = tokenizer.decode([x for x in t_last if x > 0])
print(f"\n最后一块前200字符: {decoded_last[:200]}...")
print(f"{'='*50}")